# Exercise 6: Zero-shot transfer evaluation

*Neural Morphology · version 0.1.0*

> Apply the BERTurk tagger from the Hands-On section to the whole Azerbaijani TueCL UD test set, which translates the same sentences as the Turkish TueCL set; note whether any of those Turkish sentences were used during development. Report full-bundle accuracy and per-feature accuracy by category, and check whether the two treebanks annotate the same features in the same way. Which categories transfer best? Do the results support the hypotheses in the chapter's transfer table?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "transformers==5.3.0" sentencepiece protobuf scikit-learn stanza requests` (the compiled `hfst` package needs Python 3.11 or 3.12). Tokenisers are loaded from Hugging Face, or from a local copy in TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/<org>--<name>`). Universal Dependencies files are read from the UD GitHub repositories (master branch), so internet access is needed. Cells whose model cannot be loaded print the reason instead of a result.

The BERTurk tagger of the Hands-On section is not trained in the book, so the worked example uses a Turkish tagger that exists: the Stanza IMST model, trained on Turkish only. The same evaluation code applies to the BERTurk tagger once it is trained. The model receives the gold words (`tokenize_pretokenized=True`), so tokenisation errors do not enter the scores. The Turkish TueCL test set, which translates the same sentences, is scored as an in-language reference; neither TueCL set was used to train the IMST model.

In [ ]:
import os, re, collections, statistics
from pathlib import Path
import requests

UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/master/{file}"
WORD = re.compile(r"[^\W_]+")                         # the book's word definition (Scripts and Encoding)

def ud_sentences(repo, file):
    """CoNLL-U sentences as lists of (form, lemma, upos, feats) for integer-ID words, with the sent_id."""
    text = requests.get(UD.format(repo=repo, file=file), timeout=120)
    text.raise_for_status()
    sents, words, sid = [], [], None
    for line in text.text.splitlines():
        if line.startswith("# sent_id"):
            sid = line.split("=", 1)[1].strip()
        elif not line.strip():
            if words:
                sents.append((sid, words))
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append((c[1], c[2], c[3], c[5]))
    if words:
        sents.append((sid, words))
    return sents

def load_tokenizer(name):
    """A Hugging Face tokenizer, from TurkicNLP's local model folder if present, otherwise from the Hub."""
    from transformers import AutoTokenizer
    local = Path("~/.turkicnlp/models/huggingface").expanduser() / name.replace("/", "--")
    return AutoTokenizer.from_pretrained(str(local) if (local / "tokenizer_config.json").exists() else name)

def pieces(tok, word):
    """Subword pieces of one word without special tokens and without the word-initial marker."""
    out = []
    for p in tok.tokenize(word):
        p = p.replace("▁", "").replace("Ġ", "")
        out.append(p[2:] if p.startswith("##") else p)
    return [p for p in out if p]

CORE_FEATS = ("Case", "Number", "Number[psor]", "Person", "Person[psor]", "Tense", "VerbForm", "Voice")
DEFAULTS = {("Case", "Nom"), ("Number", "Sing"), ("Person", "3")}

def core(upos, feats):
    """UPOS plus the core features, without the unmarked defaults Case=Nom, Number=Sing and Person=3,
    and with Polarity only if negative. Treebanks differ in whether they write the defaults out."""
    kv = dict(f.split("=", 1) for f in (feats or "").split("|") if "=" in f)
    keep = {k: v for k, v in kv.items() if k in CORE_FEATS and (k, v) not in DEFAULTS}
    if kv.get("Polarity") == "Neg":
        keep["Polarity"] = "Neg"
    return upos + "|" + "|".join(f"{k}={v}" for k, v in sorted(keep.items()))

In [ ]:
try:
    import stanza
    from turkicnlp.resources.registry import ModelRegistry
    nlp = stanza.Pipeline("tr", processors="tokenize,pos", tokenize_pretokenized=True, verbose=False,
                          model_dir=str(ModelRegistry.default_dir() / "stanza"), download_method=None)
except Exception as exc:
    nlp = None
    print("Stanza not available here:", type(exc).__name__)

def feats(s):
    return dict(f.split("=", 1) for f in (s or "").split("|") if "=" in f)

def evaluate(repo, file, label):
    sents = ud_sentences(repo, file)
    doc = nlp([[f for f, *_ in ws] for _, ws in sents])
    gold = [w for _, ws in sents for w in ws]
    pred = [w for s in doc.sentences for w in s.words]
    assert len(gold) == len(pred)
    rows = [(g, p) for g, p in zip(gold, pred) if g[2] != "PUNCT"]
    upos = sum(g[2] == p.upos for g, p in rows) / len(rows)
    full = sum(feats(g[3]) == feats(p.feats) for g, p in rows) / len(rows)
    per = {}
    for cat in ("Case", "Number", "Person", "Number[psor]", "Person[psor]", "Tense", "Aspect", "Mood", "VerbForm", "Voice", "Polarity", "Evident"):
        rel = [(feats(g[3]).get(cat), feats(p.feats).get(cat)) for g, p in rows if cat in feats(g[3]) or cat in feats(p.feats)]
        if rel:
            per[cat] = (sum(a == b for a, b in rel) / len(rel), len(rel))
    core_acc = sum(core(g[2], g[3]) == core(p.upos, p.feats) for g, p in rows) / len(rows)
    print(f"{label}: {len(sents)} sentences, {len(rows)} words; UPOS {upos:.3f}; full FEATS {full:.3f}; "
          f"core bundle {core_acc:.3f}")
    print("   per feature (accuracy, words with the feature in gold or prediction):",
          {k: (round(a, 3), n) for k, (a, n) in per.items()})
    return collections.Counter(k for g, _ in rows for k in feats(g[3]))

def schema(sents, label):
    nouns = [w for _, ws in sents for w in ws if w[2] == "NOUN"]
    verbs = [w for _, ws in sents for w in ws if w[2] == "VERB"]
    print(f"{label}: NOUN with Person {sum('Person' in feats(w[3]) for w in nouns) / len(nouns):.0%}, "
          f"VERB with Polarity=Pos {sum(feats(w[3]).get('Polarity') == 'Pos' for w in verbs) / len(verbs):.0%}, "
          f"VERB with Evident {sum('Evident' in feats(w[3]) for w in verbs) / len(verbs):.0%}")

schema(ud_sentences("UD_Turkish-IMST", "tr_imst-ud-train.conllu"), "IMST train (the model's training data)")
schema(ud_sentences("UD_Turkish-TueCL", "tr_tuecl-ud-test.conllu"), "Turkish TueCL")
schema(ud_sentences("UD_Azerbaijani-TueCL", "az_tuecl-ud-test.conllu"), "Azerbaijani TueCL")

if nlp:
    tr_cats = evaluate("UD_Turkish-TueCL", "tr_tuecl-ud-test.conllu", "Turkish TueCL (in-language reference)")
    az_cats = evaluate("UD_Azerbaijani-TueCL", "az_tuecl-ud-test.conllu", "Azerbaijani TueCL (zero-shot)")
    print("features used in Azerbaijani gold but not in Turkish gold:", sorted(set(az_cats) - set(tr_cats)))
    print("features used in Turkish gold but not in Azerbaijani gold:", sorted(set(tr_cats) - set(az_cats)))

## Worked solution

Output of `experiments/ch07_exercises/run_ch07_exercises.py` on 19 October 2026 (UD TueCL test sets, GitHub master branch; Stanza 1.15.0 with the IMST models):

```
IMST train (the model's training data): NOUN with Person 100%, VERB with Polarity=Pos 90%, VERB with Evident 6%
Turkish TueCL: NOUN with Person 0%, VERB with Polarity=Pos 0%, VERB with Evident 69%
Azerbaijani TueCL: NOUN with Person 4%, VERB with Polarity=Pos 0%, VERB with Evident 9%
Turkish TueCL (in-language reference): 148 sentences, 736 words; UPOS 0.807; full FEATS 0.159; core bundle 0.628
   per feature (accuracy, words with the feature in gold or prediction): {'Case': (0.914, 409), 'Number': (0.858, 598), 'Person': (0.347, 551), 'Number[psor]': (0.75, 92), 'Person[psor]': (0.804, 92), 'Tense': (0.695, 246), 'Aspect': (0.573, 246), 'Mood': (0.598, 246), 'VerbForm': (0.662, 80), 'Voice': (0.474, 19), 'Polarity': (0.142, 212), 'Evident': (0.124, 153)}
Azerbaijani TueCL (zero-shot): 148 sentences, 730 words; UPOS 0.585; full FEATS 0.099; core bundle 0.399
   per feature (accuracy, words with the feature in gold or prediction): {'Case': (0.635, 485), 'Number': (0.766, 644), 'Person': (0.25, 624), 'Number[psor]': (0.49, 96), 'Person[psor]': (0.478, 92), 'Tense': (0.544, 217), 'Aspect': (0.282, 202), 'Mood': (0.462, 210), 'VerbForm': (0.206, 68), 'Voice': (0.133, 15), 'Polarity': (0.053, 171), 'Evident': (0.2, 30)}
features used in Azerbaijani gold but not in Turkish gold: ['Degree']
features used in Turkish gold but not in Azerbaijani gold: ['Reflex']
```

**The scores.** On the Turkish TueCL test set, the Stanza tagger trained on Turkish IMST gets 80.7% UPOS, but only 15.9% of the full feature bundles right. On the Azerbaijani TueCL set, the same sentences in Azerbaijani, it gets 58.5% UPOS and 9.9% full bundles. Measured on the convention-neutral core bundle of Exercise 4, the figures are 62.8% (Turkish) and 39.9% (Azerbaijani zero-shot).

**Most of the full-bundle gap is annotation convention, not language.** The schema check shows that IMST writes `Person` on every noun and `Polarity=Pos` on 90% of verbs; neither TueCL treebank does either. Even the two TueCL treebanks differ: `Evident` appears on 69% of Turkish but 9% of Azerbaijani verbs, Azerbaijani uses `Degree`, which the Turkish set does not, and Turkish uses `Reflex`. Per-feature scores for Person, Polarity and Evident are therefore low for both languages and say little about transfer. Neither TueCL set is part of the IMST training data, so no test sentence was seen in training.

**Which categories transfer.** Comparing Azerbaijani with the Turkish reference on the same sentences: Case (0.914 → 0.635) and Number (0.858 → 0.766) keep most of their accuracy, possessor person and number keep about two thirds (0.80 → 0.48), Tense drops less (0.695 → 0.544) than Aspect, VerbForm and Voice, which fall to 0.13–0.28. This fits the chapter's transfer table for nominal morphology (case and number transfer best) and for tense, aspect and voice, which it expects to transfer only partly. Person agreement, which the table expects to transfer well, cannot be judged here: its scores are dominated by the convention difference above. With 148 sentences, one model and conventions that differ between treebanks, this is a diagnostic, not an estimate of transfer quality.

## Your turn — extension

1. Train the BERTurk tagger of the Hands-On section and rerun this cell with it.
2. Convert the Azerbaijani input to the Turkish alphabet where the letters differ (*ə* → *e*, *x* → *h*, *q* → *k* …) and measure whether the scores change. Which conversions are lossy?